In [1]:
# Global imports
import biosteam as bst, thermosteam as tmo, biorefineries as bf, numpy as np, pandas as pd
from biorefineries import cellulosic
from biosteam import main_flowsheet as F, units


bst.settings.set_thermo(['Water', 'Methanol', 'Glycerol'], cache=True)
feed = bst.Stream('feed', flow=(80, 100, 25))
bp = feed.bubble_point_at_P()
feed.T = bp.T # Feed at bubble point T
D1 = bst.units.BinaryDistillation('D1', ins=feed,
                        outs=('distillate', 'bottoms_product'),
                        LHK=('Methanol', 'Water'),
                        y_top=0.99, x_bot=0.01, k=2,
                        is_divided=True)
D1.simulate()

In [1]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')

import biosteam as bst

# Xseparation_agent's modules import each other with bare same-directory
# imports (e.g. `from sweep_separation import ...`), which only resolve
# when that directory is on sys.path -- true when running a script from
# inside it directly, but not when importing it as a package from here.

from atj_saf.atj_bst.Xseparation_agent.optimizer import optimize_reflux_ratio
from atj_saf.atj_bst.Xseparation_agent.separation_plots import (
    plot_purity_vs_reflux,
    plot_utility_cost_vs_reflux,
    plot_reflux_sweep,
)

bst.settings.set_thermo(['Water', 'Methanol', 'Glycerol'], cache=True)

feed = bst.Stream('feed', flow=(80, 100, 25), units='kmol/hr')
feed.T = feed.bubble_point_at_P().T

# k multipliers over minimum reflux, not absolute L/D values.
reflux_ratios_k = [1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0, 3.25, 3.5]

result = optimize_reflux_ratio(
    feed=feed,
    LHK=('Methanol', 'Water'),
    reflux_ratios_k=reflux_ratios_k,
    P=101325,
    spec='purity',
    target='top',
    purity_target=0.99,
    csv_path='demo_reflux_ratio_sweep.csv',
)

df = result['sweep_df']
print(df)

print(f"\nn_feasible: {result['n_feasible']}/{result['n_total']}")
print(result['message'])
if result['found']:
    print('\nBest design:')
    for k, v in result['best_design'].items():
        print(f'  {k}: {v}')

# 1. Call each plot function individually, saving each to its own file.
plot_purity_vs_reflux(df, save_path='demo_purity_vs_reflux.png', show=False)
plot_utility_cost_vs_reflux(df, save_path='demo_utility_cost_vs_reflux.png', show=False)

# 2. Or use the convenience wrapper to draw both at once.
plot_reflux_sweep(df, save_dir='.', show=False)

print('\nSaved: demo_purity_vs_reflux.png, demo_utility_cost_vs_reflux.png, '
      'purity_vs_reflux.png, utility_cost_vs_reflux.png')


ModuleNotFoundError: No module named 'sweep_separation'

In [1]:
from separation_trial import run_separation

In [1]:
import biosteam as bst
from separation_trial import run_separation


# ---------------------------------------------------------
# 1. Set up the thermodynamic chemicals
# ---------------------------------------------------------

bst.settings.set_thermo(
    ['Water', 'Methanol', 'Glycerol'],
    cache=True
)


# ---------------------------------------------------------
# 2. Create the feed stream
# ---------------------------------------------------------

feed = bst.Stream(
    'feed',
    flow=(80, 100, 25),
    units='kmol/hr'
)

In [2]:
result = run_separation(
    feed=feed,
    LHK=('Methanol', 'Water'),

    # Operating condition
    reflux_ratio=2.0,
    P=101325,

    # Separation specification
    spec='purity',
    target='top',

    # Distillation specifications
    y_top=0.99,
    x_bot=0.01,
)


In [4]:
for R in [1.5, 2.0, 2.5, 3.0, 3.5]:

    result = run_separation(
        feed=feed,
        LHK=('Methanol', 'Water'),
        reflux_ratio=R,
        P=101325,
        spec='purity',
        target='top',
        y_top=0.99,
        x_bot=0.01,
    )

    print(
        R,
        result['purity']['achieved'],
        result['capex_usd'],
        result['utilities']['heating_cost_USD_per_hr'],
        result['utilities']['cooling_cost_USD_per_hr'],
        result['feasible'],
    )

1.5 0.99 211042.21137500647 54.34970564844269 1.2062093867131205 True
2.0 0.99 214962.36907292466 62.220559021843364 1.6082793299521048 True
2.5 0.99 216385.97282171948 71.19541409894285 2.010349299946949 True
3.0 0.99 225905.79389770923 80.17227001588417 2.4124192876304904 True
3.5 0.99 229548.44606890364 89.1507776165911 2.814489287618261 True


C:\Users\hwadg\AppData\Local\Temp\ipykernel_36588\2336271716.py:3: RuntimeWarning: <BinaryDistillation: D1> has been replaced in registry
  result = run_separation(
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: distillate> has been replaced in registry
  self._register(ID)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\_stream.py:398: RuntimeWarning: <Stream: bottoms> has been replaced in registry
  self._register(ID)


In [2]:
D1.show(T='degC', P='atm', composition=True)

BinaryDistillation: D1
ins...
[0] feed  
    phase: 'l', T: 76.082 degC, P: 1 atm
    composition (%): Water     39
                     Methanol  48.8
                     Glycerol  12.2
                     --------  205 kmol/hr
outs...
[0] distillate  
    phase: 'g', T: 64.854 degC, P: 1 atm
    composition (%): Water     1
                     Methanol  99
                     --------  100 kmol/hr
[1] bottoms_product  
    phase: 'l', T: 100.02 degC, P: 1 atm
    composition (%): Water     75.4
                     Methanol  0.761
                     Glycerol  23.9
                     --------  105 kmol/hr
